In [1]:
# Tensorflow has lots of warnings that I don't like. This part forces the code ignore them.
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0=all, 1=filter INFO, 2=filter WARNING, 3=filter ERROR too

# Use the second GPU.
os.environ['CUDA_VISIBLE_DEVICES'] = '1'

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.layers import Dense, Add, Input, Activation, Lambda, BatchNormalization, Conv1D, GlobalAveragePooling1D
from sklearn.metrics import confusion_matrix,accuracy_score
from sklearn.model_selection import train_test_split
from model import LogGaussMF
from tensorflow.keras.regularizers import l2
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, LearningRateScheduler
from tensorflow.keras.callbacks import ReduceLROnPlateau


# Read data and split it
number_of_classes = 6
with open('./data/emulator_data_x.npy', 'rb') as f:
    x_data = np.load(f)
    
with open('./data/emulator_data_y.npy', 'rb') as f:
    y_data = np.load(f)
    
    
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.10,stratify=y_data,random_state=42)

#data normalization
x_test = (x_test-np.min(x_train))/(np.max(x_train)-np.min(x_train))
x_train = (x_train-np.min(x_train))/(np.max(x_train)-np.min(x_train))

# Convert class vectors to binary class matrices.
y_true_test = y_test
y_true_train = y_train

# Data preprocessing for neural network training
y_train = tf.keras.utils.to_categorical(y_train, number_of_classes)
y_test = tf.keras.utils.to_categorical(y_test, number_of_classes)

# Input image dimensions.
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)
input_shape = x_train.shape[1:]

In [2]:
def resnet_layer(inputs,
                 num_filters=16,
                 kernel_size=3,
                 strides=1,
                 activation='relu',
                 batch_normalization=True,
                 conv_first=True):
    conv = Conv1D(num_filters,
                  kernel_size=kernel_size,
                  strides=strides,
                  padding='same',
                  kernel_initializer='he_normal',
                  kernel_regularizer=l2(1e-4))

    x = inputs
    if conv_first:
        x = conv(x)
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
    else:
        if batch_normalization:
            x = BatchNormalization()(x)
        if activation is not None:
            x = Activation(activation)(x)
        x = conv(x)
    return x


In [3]:
def resnet_backend_v2(input_shape, depth, num_classes=10):
    if (depth - 2) % 9 != 0:
        raise ValueError('depth should be 9n+2 (eg 56 or 110 in [b])')
    # Start model definition.
    num_filters_in = 16
    num_res_blocks = int((depth - 2) / 9)

    inputs = Input(shape=input_shape)
    # v2 performs Conv2D with BN-ReLU on input before splitting into 2 paths
    x = resnet_layer(inputs=inputs,
                     num_filters=num_filters_in,
                     conv_first=True)

    # Instantiate the stack of residual units
    for stage in range(3):
        for res_block in range(num_res_blocks):
            activation = 'relu'
            batch_normalization = True
            strides = 1
            if stage == 0:
                num_filters_out = num_filters_in * 4
                if res_block == 0:  # first layer and first stage
                    activation = None
                    batch_normalization = False
            else:
                num_filters_out = num_filters_in * 2
                if res_block == 0:  # first layer but not first stage
                    strides = 2    # downsample

            # bottleneck residual unit
            y = resnet_layer(inputs=x,
                             num_filters=num_filters_in,
                             kernel_size=1,
                             strides=strides,
                             activation=activation,
                             batch_normalization=batch_normalization,
                             conv_first=False)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters_in,
                             conv_first=False)
            y = resnet_layer(inputs=y,
                             num_filters=num_filters_out,
                             kernel_size=1,
                             conv_first=False)
            if res_block == 0:
                # linear projection residual shortcut connection to match
                # changed dims
                x = resnet_layer(inputs=x,
                                 num_filters=num_filters_out,
                                 kernel_size=1,
                                 strides=strides,
                                 activation=None,
                                 batch_normalization=False)
            x = tf.keras.layers.add([x, y])

        num_filters_in = num_filters_out

    # Add classifier on top.
    # v2 has BN-ReLU before Pooling
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    features = GlobalAveragePooling1D()(x)

    return inputs, features

In [4]:

# define the resnet n6 backend for cnn feature extraction
# note that the backend is using the new defined resnet_layer which uses Conv1D
def backend(input_shape):
    n = 6
    depth = n * 9 + 2
    model_type = 'ResNet%dv%d-trial-1' % (depth, 2)
    inputs, features = resnet_backend_v2(input_shape=input_shape,depth=depth)
    
    return inputs, features


# add fuzzy classifier to it
inputs, features = backend(input_shape=input_shape)
memberships = LogGaussMF(number_of_classes)(features)
rules = Lambda(lambda x: tf.keras.ops.sum(x, axis=-1),output_shape=(number_of_classes,))(memberships)
linear = Dense(number_of_classes)(features)
logits = Add()([rules, linear])
outputs = Activation("softmax")(logits)
model = Model(inputs=inputs, outputs=outputs)

# callbacks = get_callbacks("EPCOR")

model.compile(
    loss='categorical_crossentropy',
    optimizer=tf.keras.optimizers.Adam(),
    metrics=['accuracy'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 350, 1)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 350, 16)   │         64 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 350, 16)   │         64 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 350, 16)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 350, 16)   │        272 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 350, 16)   │         64 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 350, 16)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 350, 16)   │        784 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 350, 16)   │         64 │ conv1d_2[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 350, 16)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_4 (Conv1D)   │ (None, 350, 64)   │      1,088 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_3 (Conv1D)   │ (None, 350, 64)   │      1,088 │ activation_2[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 350, 64)   │          0 │ conv1d_4[0][0],   │
│                     │                   │            │ conv1d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 350, 64)   │        256 │ add[0][0]         │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_3        │ (None, 350, 64)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_5 (Conv1D)   │ (None, 350, 16)   │      1,040 │ activation_3[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 350, 16)   │         64 │ conv1d_5[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 350, 16)   │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_6 (Conv1D)   │ (None, 350, 16)   │        784 │ activation_4[0][

 Total params: 928,902 (3.54 MB)

 Trainable params: 918,502 (3.50 MB)

 Non-trainable params: 10,400 (40.62 KB)

In [9]:

def lr_schedule(epoch):
    """Learning Rate Schedule

    Learning rate is scheduled to be reduced after 80, 120, 160, 180 epochs.
    Called automatically every epoch as part of callbacks during training.

    # Arguments
        epoch (int): The number of epochs

    # Returns
        lr (float32): learning rate
    """
    lr = 1e-3
    if epoch > 180:
        lr *= 0.5e-3
    elif epoch > 160:
        lr *= 1e-3
    elif epoch > 120:
        lr *= 1e-2
    elif epoch > 5:
        lr *= 1e-1
    print('Learning rate: ', lr)
    return lr


def get_callbacks(model_type):
    save_dir = os.path.join(os.getcwd(), 'Saved_Models/')
    model_name = "%s_model.{epoch:03d}.keras" % model_type
    if not os.path.isdir(save_dir):
        os.makedirs(save_dir)
    filepath = os.path.join(save_dir, model_name)

    # Prepare callbacks for model saving and for learning rate adjustment.
    checkpoint = ModelCheckpoint(
        filepath=filepath,
        monitor='val_accuracy',
        verbose=1,
        save_best_only=True,mode='max')

    lr_scheduler = LearningRateScheduler(lr_schedule)

    lr_reducer = ReduceLROnPlateau(
        factor=np.sqrt(0.1),
        cooldown=0,
        patience=5,
        min_lr=0.5e-6)

    return [checkpoint, lr_reducer, lr_scheduler]    

In [10]:
model_type = "DCNFIS_CM_ResNetV2_n6"
callbacks = get_callbacks(model_type)

In [11]:
# x_train, x_test, y_train, y_test
model.fit(x_train, y_train,
          batch_size=128,
          epochs=20,
          validation_data=(x_test, y_test),
          shuffle=True,
          callbacks=callbacks)

Learning rate:  0.001
Epoch 1/20
95/97 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9641 - loss: 0.4101
Epoch 1: val_accuracy improved from None to 0.96635, saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_CM_ResNetV2_n6_model.001.keras

Epoch 1: finished saving model to /home/mojtaba/Desktop/TF_Works/Thesis_Rep/Saved_Models/DCNFIS_CM_ResNetV2_n6_model.001.keras
97/97 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - accuracy: 0.9642 - loss: 0.4096 - val_accuracy: 0.9663 - val_loss: 0.6721 - learning_rate: 0.0010
Learning rate:  0.001
Epoch 2/20
95/97 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9657 - loss: 0.3789
Epoch 2: val_accuracy did not improve from 0.96635
97/97 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.9656 - loss: 0.3791 - val_accuracy: 0.7813 - val_loss: 1.1304 - learning_rate: 0.0010
Learning rate:  0.001
Epoch 3/20
97/97 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9585 - loss: 0.3854
Epoch 3: val_accuracy did not improve from 0.96635
97/97 ━

In [13]:
model.load_weights('./Saved_Models/DCNFIS_CM_ResNetV2_n6_model.015.keras')
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

Test loss: 0.29168015718460083
Test accuracy: 0.980248749256134
